In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Qwant07/rl-eeg-cursor.git /content/rl-eeg-cursor
%cd /content/rl-eeg-cursor
!pip install -q mne scikit-learn

In [ ]:
import os, shutil

DRIVE_DATA = '/content/drive/MyDrive/bci/preprocessed'
os.makedirs('preprocessed', exist_ok=True)
for fname in ['S01_preprocessed.h5', 'S05_preprocessed.h5']:
    src = os.path.join(DRIVE_DATA, fname)
    dst = os.path.join('preprocessed', fname)
    if not os.path.exists(dst):
        os.symlink(src, dst)

src = '/content/drive/MyDrive/bci/results'
if os.path.exists(src):
    shutil.copytree(src, 'results', dirs_exist_ok=True)
    print('Restored existing results from Drive')

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# S01 EEGNet — sessions 1+2 (6.8 GB fits in Colab), AR-only (default)
!python -m src.decoders.train \
    --subject S01 --model eegnet --device cuda \
    --epochs 200 --batch_size 128 --patience 20 \
    --train_sessions 1 2 --num_workers 2 --out_dir results

In [ ]:
# S01 LSTM — sessions 1+2, AR-only (default)
!python -m src.decoders.train \
    --subject S01 --model lstm --device cuda \
    --epochs 200 --batch_size 128 --patience 20 \
    --train_sessions 1 2 --num_workers 2 --out_dir results

In [ ]:
# S05 EEGNet — session 1 only (15 GB, RAM limit), AR-only (default)
!python -m src.decoders.train \
    --subject S05 --model eegnet --device cuda \
    --epochs 200 --batch_size 128 --patience 20 \
    --train_sessions 1 --num_workers 2 --out_dir results

In [ ]:
# S05 LSTM — session 1 only, AR-only (default)
!python -m src.decoders.train \
    --subject S05 --model lstm --device cuda \
    --epochs 200 --batch_size 128 --patience 20 \
    --train_sessions 1 --num_workers 2 --out_dir results

In [ ]:
import json, gc
import numpy as np
import h5py
from pathlib import Path
from src.baselines.lda_decoder import BandPowerLDA

def load_ar_sessions(h5_path, subject, sessions):
    """Load AR-only runs from HDF5."""
    X_parts, y_parts = [], []
    with h5py.File(h5_path, 'r') as f:
        subj = f[subject]
        for s in sessions:
            sk = f'session_{s:02d}'
            if sk not in subj or 'AR' not in subj[sk]:
                continue
            for rk in sorted(subj[sk]['AR']):
                X_parts.append(subj[sk]['AR'][rk]['X'][:])
                y_parts.append(subj[sk]['AR'][rk]['y'][:])
    return np.concatenate(X_parts), np.concatenate(y_parts)

train_sessions = {'S01': [1, 2], 'S05': [1]}

for subject in ['S01', 'S05']:
    h5_path = f'preprocessed/{subject}_preprocessed.h5'
    print(f'\n=== LDA Baseline (AR-only): {subject} ===')
    X_train, y_train = load_ar_sessions(h5_path, subject, train_sessions[subject])
    X_val, y_val = load_ar_sessions(h5_path, subject, [3])
    print(f'Train: {X_train.shape[0]} epochs, Val: {X_val.shape[0]} epochs')

    lda = BandPowerLDA(fs=1000.0)
    lda.fit(X_train, y_train)
    scores = lda.score(X_val, y_val)
    print(f"  R²  vx={scores['r2_vx']:.4f}  vy={scores['r2_vy']:.4f}  mean={scores['r2_mean']:.4f}")
    print(f"  NMSE vx={scores['nmse_vx']:.4f} vy={scores['nmse_vy']:.4f} mean={scores['nmse_mean']:.4f}")

    out = Path('results') / subject / 'lda'
    out.mkdir(parents=True, exist_ok=True)
    with open(out / 'best_metrics.json', 'w') as f:
        json.dump(scores, f, indent=2)
    print(f'  Saved -> {out / "best_metrics.json"}')

    del X_train, y_train, X_val, y_val, lda
    gc.collect()

In [ ]:
import json
from pathlib import Path

print(f'{"Subject":<10} {"Model":<10} {"R² vx":>8} {"R² vy":>8} {"R² mean":>8} {"NMSE mean":>10}')
print('-' * 60)
for subject in ['S01', 'S05']:
    for model in ['lda', 'eegnet', 'lstm']:
        p = Path('results') / subject / model / 'best_metrics.json'
        if not p.exists():
            print(f'{subject:<10} {model:<10} — not found —')
            continue
        with open(p) as f:
            m = json.load(f)
        print(f'{subject:<10} {model:<10} {m["r2_vx"]:>8.4f} {m["r2_vy"]:>8.4f} {m["r2_mean"]:>8.4f} {m["nmse_mean"]:>10.4f}')

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for col, subject in enumerate(['S01', 'S05']):
    for model, color in [('eegnet', 'tab:blue'), ('lstm', 'tab:orange')]:
        path = Path('results') / subject / model / 'history.json'
        if not path.exists():
            continue
        with open(path) as f:
            h = json.load(f)
        epochs = range(1, len(h['train_loss']) + 1)
        axes[0, col].plot(epochs, h['train_loss'], '--', color=color, alpha=0.5, label=f'{model} train')
        axes[0, col].plot(epochs, h['val_loss'], '-', color=color, label=f'{model} val')
        axes[0, col].set_title(f'{subject} — Loss')
        axes[0, col].set_xlabel('Epoch')
        axes[0, col].set_ylabel('MSE Loss')
        axes[0, col].legend()
        axes[1, col].plot(epochs, h['val_r2_mean'], '-', color=color, label=f'{model}')
        axes[1, col].set_title(f'{subject} — Validation R²')
        axes[1, col].set_xlabel('Epoch')
        axes[1, col].set_ylabel('R² (mean)')
        axes[1, col].legend()
plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150)
plt.show()

In [ ]:
import shutil
shutil.copytree('results', '/content/drive/MyDrive/bci/results', dirs_exist_ok=True)
print('All results saved to Drive')